# Persona Compound Fairness — Consolidated Triage Table (Table 1)

Aggregates over- and under-triage rates for the four doctor models across the five personae (Default, Anxious, Dismissive, Limited, Informed) into a single table, and emits the LaTeX used in Supplementary Material 7.

Each model is one run directory under `../results/runs`. The per-model run IDs match the individual `results_*.ipynb` notebooks in this folder. Over-triage is computed over cases eligible for over-triage (gold ≠ urgent, N=721); under-triage over cases eligible for under-triage (gold ≠ self-care, N=964). Each persona is evaluated on the same 1,164 cases.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

# --- Configuration ---
# Display name -> run id. Row order matches Table 1 in Supplementary Material 7
# (the document's canonical model order; Claude is last as it lacks probability metrics).
RUNS = {
    "Gemini 3.5 Flash": "persona_20260601_183211_d415d86a",
    "GPT-5.5": "persona_20260604_135655_333650b8",
    "GPT-5.4-mini": "persona_20260521_172943_d0326dbc",
    "Claude Opus 4.6": "persona_20260601_142913_bf3804f5",
}
RUNS_DIR = Path("../results/runs")

TRIAGE_ORD = {"self-care": 0, "non-urgent": 1, "urgent": 2}

# Map raw condition names to the persona labels used in the table.
PERSONA_FULL = {
    "baseline": "Default",
    "anxious_patient": "Anxious",
    "dismissive_patient": "Dismissive",
    "limited_communicator": "Limited",
    "informed_advocate": "Informed",
}
PERSONA_ORDER = ["Default", "Anxious", "Dismissive", "Limited", "Informed"]

In [ ]:
import numpy as np


def load_run(run_id):
    """Load AV+HV conversations for a run and majority-vote the predicted triage."""
    rows = []
    for p in sorted((RUNS_DIR / run_id / "conversations").glob("*.json")):
        with open(p) as f:
            r = json.load(f)
        if r.get("dataset_name") not in ("av", "hv"):
            continue
        votes = [a["urgency"] for a in r.get("assessments", []) if a.get("urgency")]
        pred = Counter(votes).most_common(1)[0][0] if votes else None
        rows.append({
            "cond": r.get("condition_name"),
            "gold": r.get("gold_triage"),
            "pred": pred,
        })
    df = pd.DataFrame(rows)
    df = df[df["pred"].notna() & df["gold"].notna()].copy()
    df["gold_ord"] = df["gold"].map(TRIAGE_ORD)
    df["pred_ord"] = df["pred"].map(TRIAGE_ORD)
    return df[df["gold_ord"].notna() & df["pred_ord"].notna()].copy()


def bootstrap_ci(indicators, n_boot=10_000, seed=42):
    """95% percentile bootstrap CI for the mean of a 0/1 indicator array."""
    arr = np.asarray(indicators, dtype=float)
    if arr.size == 0:
        return (0.0, 0.0)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    boots = arr[idx].mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(lo), float(hi)


def persona_rates(df):
    """Return {persona: (rate%, lo%, hi%)} for over- and under-triage, plus eligible Ns.

    Rates and CIs are in percentage points.
    """
    over, under = {}, {}
    n_over = n_under = 0
    for cond, grp in df.groupby("cond"):
        ot = grp[grp["gold"] != "urgent"]      # eligible for over-triage
        ut = grp[grp["gold"] != "self-care"]   # eligible for under-triage
        ot_ind = (ot["pred_ord"] > ot["gold_ord"]).astype(float).to_numpy()
        ut_ind = (ut["pred_ord"] < ut["gold_ord"]).astype(float).to_numpy()
        ot_lo, ot_hi = bootstrap_ci(ot_ind)
        ut_lo, ut_hi = bootstrap_ci(ut_ind)
        over[PERSONA_FULL[cond]] = (ot_ind.mean() * 100, ot_lo * 100, ot_hi * 100)
        under[PERSONA_FULL[cond]] = (ut_ind.mean() * 100, ut_lo * 100, ut_hi * 100)
        n_over, n_under = len(ot), len(ut)
    return over, under, n_over, n_under

In [ ]:
over, under = {}, {}  # nested: {model: {persona: (rate%, lo%, hi%)}}
n_over = n_under = None
for model, run_id in RUNS.items():
    df = load_run(run_id)
    o, u, n_over, n_under = persona_rates(df)
    over[model] = o
    under[model] = u
    print(f"{model}: loaded {len(df)} conversations")

print(f"\nOver-triage eligible N = {n_over}; Under-triage eligible N = {n_under}")


def fmt_cell(t):
    """Format (rate, lo, hi) as 'rate (lo, hi)' with no percent sign."""
    return f"{t[0]:.1f} ({t[1]:.1f}, {t[2]:.1f})"


over_fmt = pd.DataFrame(
    {m: {p: fmt_cell(over[m][p]) for p in PERSONA_ORDER} for m in RUNS}
).T[PERSONA_ORDER]
under_fmt = pd.DataFrame(
    {m: {p: fmt_cell(under[m][p]) for p in PERSONA_ORDER} for m in RUNS}
).T[PERSONA_ORDER]

In [ ]:
print(f"=== Over-triage, % (n = {n_over}) — rate (95% bootstrap CI) ===")
display(over_fmt)
print(f"\n=== Under-triage, % (n = {n_under}) — rate (95% bootstrap CI) ===")
display(under_fmt)

In [ ]:
# --- Emit LaTeX for Supplementary Material 7 (Table 1) ---
# This mirrors render_consolidated_triage_table() in
# supplements/7_fairness/build_report.py (the source of truth for the PDF).
N_CASES = 1164
N_MODELS = len(RUNS)
N_PERSONAS = len(PERSONA_ORDER)


def panel_rows(data):
    lines = []
    for model in RUNS:
        cells = " & ".join(
            f"{data[model][p][0]:.1f} ({data[model][p][1]:.1f}, {data[model][p][2]:.1f})"
            for p in PERSONA_ORDER
        )
        lines.append(f"{model} & {cells} \\\\")
    return "\n".join(lines)


header = " & ".join(["\\textbf{Model}"] + [f"\\textbf{{{p}}}" for p in PERSONA_ORDER])
span = len(PERSONA_ORDER) + 1

TPL = r"""\tblcaption{Over- and under-triage rates (\%) with 95\% bootstrap confidence intervals for the __NMODELS__ doctor models across the __NPERSONAS__ personae.}
\begin{center}
{\footnotesize\setlength{\tabcolsep}{4pt}
\begin{tabular}{lccccc}
\toprule
__HEADER__ \\
\midrule
\multicolumn{__SPAN__}{l}{\textit{Over-triage (n=__NOVER__), \%}} \\
__OVERROWS__
\midrule
\multicolumn{__SPAN__}{l}{\textit{Under-triage (n=__NUNDER__), \%}} \\
__UNDERROWS__
\bottomrule
\end{tabular}}
\end{center}
{\footnotesize\textbf{Legend.} Each cell reports the point-estimate rate followed by its 95\% bootstrap confidence interval in parentheses, both expressed as percentages: rate (lower, upper). \textbf{Over-triage} is the percentage of over-triage-eligible cases (gold-standard label $\neq$ urgent; n=__NOVER__) assigned a \emph{higher} urgency than the gold standard. \textbf{Under-triage} is the percentage of under-triage-eligible cases (gold-standard label $\neq$ self-care; n=__NUNDER__) assigned a \emph{lower} urgency than the gold standard; under-triage is clinically more concerning as it may delay needed care. Each persona is evaluated on the same __NCASES__ cases per model. Confidence intervals are obtained by percentile bootstrap over the eligible cases (10,000 resamples, seed 42). Personas: \emph{Default} (baseline communication), \emph{Anxious} (anxious patient), \emph{Dismissive} (dismissive patient), \emph{Limited} (limited communicator), and \emph{Informed} (informed advocate); see the Personas subsection of the Experimental Design for full definitions.}"""

latex = (
    TPL.replace("__HEADER__", header)
    .replace("__SPAN__", str(span))
    .replace("__NMODELS__", str(N_MODELS))
    .replace("__NPERSONAS__", str(N_PERSONAS))
    .replace("__NOVER__", f"{n_over:,}")
    .replace("__NUNDER__", f"{n_under:,}")
    .replace("__NCASES__", f"{N_CASES:,}")
    .replace("__OVERROWS__", panel_rows(over))
    .replace("__UNDERROWS__", panel_rows(under))
)
print(latex)